# Natural language processing (NLP)

This notebook turns a short example text into numerical features. We explore cleaning, tokenization, stopword removal, lemmatization, stemming, and Bag of Words. Run the cells in order and compare the intermediate outputs.

## 1. Prepare the environment

The first cell installs Gensim in the notebook environment. Word2Vec is imported but is not trained or used later; the implemented representation is Bag of Words. The following cells also require NLTK and scikit-learn. NLTK downloads language resources, which requires internet access on the first run.

In [1]:
!pip install gensim


   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   -- ------------------------------------- 1.6/24.4 MB 8.9 MB/s eta 0:00:03
   ----- ---------------------------------- 3.4/24.4 MB 8.8 MB/s eta 0:00:03
   -------- ------------------------------- 5.2/24.4 MB 8.7 MB/s eta 0:00:03
   ---------- ----------------------------- 6.6/24.4 MB 8.1 MB/s eta 0:00:03
   -------------- ------------------------- 8.7/24.4 MB 8.4 MB/s eta 0:00:02
   --------------- ------------------------ 9.7/24.4 MB 8.0 MB/s eta 0:00:02
   ------------------ --------------------- 11.3/24.4 MB 7.8 MB/s eta 0:00:02
   -------------------- ------------------- 12.6/24.4 MB 7.7 MB/s eta 0:00:02
   ----------------------- ---------------- 14.2/24.4 MB 7.7 MB/s eta 0:00:02
   -------------------------- ------------- 16.0/24.4 MB 7.7 MB/s eta 0:00:02
   ----------------------------- ---------- 17.8/24.4 MB 7.8 MB/s eta 0:00:01
   -------------------------------- ------- 19.7/24.4 MB 7.9 MB/s eta 0:00:01


In [2]:
from gensim.models import Word2Vec


In [3]:
# Cell 1: Import libraries
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from gensim.models import Word2Vec

# Download required NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\iouzarf\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\iouzarf\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\iouzarf\AppData\Roaming\nltk_data...
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\iouzarf\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

## 2. Inspect the example text

The input consists of four sentences about students and their subjects. This small, hand-written example makes the transformations easy to follow; it is not a dataset for measuring model performance.

In [4]:
# Cell 2: Example text
text = """
Alice studies Mathematics and Physics. Bob is enrolled in Computer Science and AI.
Carol loves studying English and History. David is taking courses in Biology and Chemistry.
"""
print("Original Text:\n", text)


Original Text:
 
Alice studies Mathematics and Physics. Bob is enrolled in Computer Science and AI.
Carol loves studying English and History. David is taking courses in Biology and Chemistry.



## 3. Normalize the text

Lowercasing makes words with different capitalization share the same representation. This reduces vocabulary size, but also removes capitalization cues such as proper names.

In [5]:
# Convert to lowercase
text_clean = text.lower()
print("Cleaned Text:\n", text_clean)

Cleaned Text:
 
alice studies mathematics and physics. bob is enrolled in computer science and ai.
carol loves studying english and history. david is taking courses in biology and chemistry.



### Remove numbers and punctuation

The regular expression in the next cell removes digit sequences. The example contains no digits, so this step has no visible effect. The print statement displays `text_clean`, while the result is stored in `text_no_numbers`.

The following cell removes characters other than word characters and whitespace. Removing punctuation also removes sentence boundaries, which affects sentence tokenization later.

In [6]:
# Remove numbers
text_no_numbers = re.sub(r'\d+', '',text_clean )
print("Cleaned Text:\n", text_clean)

Cleaned Text:
 
alice studies mathematics and physics. bob is enrolled in computer science and ai.
carol loves studying english and history. david is taking courses in biology and chemistry.



In [8]:
# Cell 3: Text Preprocessing - Lowercasing, removing numbers and punctuation


# Remove punctuation
text_clean = re.sub(r'[^\w\s]', '', text_no_numbers)

print("Cleaned Text:\n", text_clean)


Cleaned Text:
 
alice studies mathematics and physics bob is enrolled in computer science and ai
carol loves studying english and history david is taking courses in biology and chemistry



## 4. Split text into tokens

Sentence tokenization divides text into sentences, and word tokenization divides each sentence into tokens. The nested list comprehension creates one list of tokens per detected sentence.

Here, sentence punctuation has already been removed, so the original four sentences are generally treated as one text segment. The second Bag of Words example later starts from the original text to preserve those boundaries.

In [9]:
# Cell 4: Tokenization
from nltk.tokenize import word_tokenize, sent_tokenize

# Sentence tokenization
sentences = sent_tokenize(text_clean)
print("Tokenized Sentences:\n", sentences)

# Word tokenization
words = [word_tokenize(sentence) for sentence in sentences]
print("\nTokenized Words:\n", words)


Tokenized Sentences:
 ['\nalice studies mathematics and physics bob is enrolled in computer science and ai\ncarol loves studying english and history david is taking courses in biology and chemistry']

Tokenized Words:
 [['alice', 'studies', 'mathematics', 'and', 'physics', 'bob', 'is', 'enrolled', 'in', 'computer', 'science', 'and', 'ai', 'carol', 'loves', 'studying', 'english', 'and', 'history', 'david', 'is', 'taking', 'courses', 'in', 'biology', 'and', 'chemistry']]


### Optional Python syntax examples

The commented examples below show a loop and an equivalent list comprehension for doubling numbers. They illustrate the syntax used in token processing and do not affect the NLP pipeline.

In [11]:
#numbers = [1, 2, 3, 4, 5]

#doubled = []
#for n in numbers:
    #doubled.append(n * 2)

#print(doubled)


[2, 4, 6, 8, 10]


In [14]:
#numbers = [1, 2, 3, 4, 5]

#doubled = [n * 2 for n in numbers]

#print(doubled)


[2, 4, 6, 8, 10]


## 5. Remove stopwords

Stopwords are common words such as ?and? or ?in?. The code filters each token list using the English stopword set. This can emphasize content words, but the choice depends on the task: removing words such as ?not? can change meaning.

In [15]:
# Cell 5: Remove Stopwords
stop_words = set(stopwords.words('english'))

words_no_stop = []
for sentence in words:
    filtered = [w for w in sentence if w not in stop_words]
    words_no_stop.append(filtered)

print("Words after Stopword Removal:\n", words_no_stop)


Words after Stopword Removal:
 [['alice', 'studies', 'mathematics', 'physics', 'bob', 'enrolled', 'computer', 'science', 'ai', 'carol', 'loves', 'studying', 'english', 'history', 'david', 'taking', 'courses', 'biology', 'chemistry']]


## 6. Compare lemmatization and stemming

Lemmatization aims to map a word to its dictionary form. This call does not supply a part of speech, so the lemmatizer uses its default noun interpretation; verbs may remain unchanged. Inspect the output instead of assuming every inflected word becomes its base form.

In [16]:
# Cell 6: Lemmatization
lemmatizer = WordNetLemmatizer()

words_lemmatized = []
for sentence in words_no_stop:
    lem_sentence = [lemmatizer.lemmatize(w) for w in sentence]
    words_lemmatized.append(lem_sentence)

print("Lemmatized Words:\n", words_lemmatized)


Lemmatized Words:
 [['alice', 'study', 'mathematics', 'physic', 'bob', 'enrolled', 'computer', 'science', 'ai', 'carol', 'love', 'studying', 'english', 'history', 'david', 'taking', 'course', 'biology', 'chemistry']]


### Stemming

Porter stemming applies rules to shorten words, and the result need not be a dictionary word. Both this step and lemmatization start from `words_no_stop`, so they are alternative transformations. The next Bag of Words example uses the stemmed version.

In [17]:
# Cell 7: Stemming
stemmer = PorterStemmer()

words_stemmed = []
for sentence in words_no_stop:
    stem_sentence = [stemmer.stem(w) for w in sentence]
    words_stemmed.append(stem_sentence)

print("Stemmed Words:\n", words_stemmed)


Stemmed Words:
 [['alic', 'studi', 'mathemat', 'physic', 'bob', 'enrol', 'comput', 'scienc', 'ai', 'carol', 'love', 'studi', 'english', 'histori', 'david', 'take', 'cours', 'biolog', 'chemistri']]


## 7. Build a Bag of Words matrix

The token lists are joined into strings before vectorization. `CountVectorizer` learns a vocabulary and counts its terms: rows represent text segments, columns represent vocabulary terms, and values represent occurrence counts. Match the displayed vocabulary to the matrix columns.

Bag of Words does not preserve word order or directly represent meaning. Because sentence boundaries were removed earlier, this first matrix generally has one row.

In [18]:
# Cell 8: Bag of Words from Stemmed Words
from sklearn.feature_extraction.text import CountVectorizer

# Join stemmed words back into sentences
sentences_stemmed = [" ".join(sentence) for sentence in words_stemmed]

# Initialize CountVectorizer
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(sentences_stemmed)

# Convert BoW matrix to array and display
print("Vocabulary:", vectorizer.get_feature_names_out())
print("BoW Matrix:\n", X_bow.toarray())


Vocabulary: ['ai' 'alic' 'biolog' 'bob' 'carol' 'chemistri' 'comput' 'cours' 'david'
 'english' 'enrol' 'histori' 'love' 'mathemat' 'physic' 'scienc' 'studi'
 'take']
BoW Matrix:
 [[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 1]]


## 8. Preserve sentences before vectorization

This second example restores the original text and detects sentences before removing punctuation. Compare the four detected sentences with the earlier tokenization output.

In [19]:
# Sample text
text = """Alice studies Mathematics and Physics. Bob is enrolled in Computer Science and AI.
Carol loves studying English and History. David is taking courses in Biology and Chemistry."""

# Import sentence tokenizer
from nltk.tokenize import sent_tokenize, word_tokenize
import nltk
nltk.download('punkt')

# Step 1: Sentence Tokenization
sentences = sent_tokenize(text)
print("Sentences:\n", sentences)


Sentences:
 ['Alice studies Mathematics and Physics.', 'Bob is enrolled in Computer Science and AI.', 'Carol loves studying English and History.', 'David is taking courses in Biology and Chemistry.']


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\iouzarf\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


### Compare the sentence-level matrix

Each original sentence now becomes a separate row. This example does not apply the earlier stopword removal or stemming. The vectorizer applies its own default text processing, so the vocabulary differs from the first example.

Check which terms occur in several rows and which identify a particular sentence. The matrix could serve as input to a later model, but this notebook does not train or evaluate a classifier.

In [20]:
# Step 1: Sentence Tokenization
sentences = sent_tokenize(text)

# Step 2: Word Tokenization and join tokens back into sentences
words_tokenized = [word_tokenize(sentence) for sentence in sentences]
sentences_for_bow = [" ".join(tokens) for tokens in words_tokenized]
# Step 3: Create Bag of Words model
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(sentences_for_bow)

# Step 4: Display vocabulary and BoW matrix
print("Vocabulary:", vectorizer.get_feature_names_out())
print("\nBoW Matrix:\n", X_bow.toarray())

Vocabulary: ['ai' 'alice' 'and' 'biology' 'bob' 'carol' 'chemistry' 'computer'
 'courses' 'david' 'english' 'enrolled' 'history' 'in' 'is' 'loves'
 'mathematics' 'physics' 'science' 'studies' 'studying' 'taking']

BoW Matrix:
 [[0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0]
 [1 0 1 0 1 0 0 1 0 0 0 1 0 1 1 0 0 0 1 0 0 0]
 [0 0 1 0 0 1 0 0 0 0 1 0 1 0 0 1 0 0 0 0 1 0]
 [0 0 1 1 0 0 1 0 1 1 0 0 0 1 1 0 0 0 0 0 0 1]]
